## Simple RAG
Here we can use the Python SDK to develop the simple rag agent, then save the agent to a config.yaml and run it from there.

In [ ]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath("../../../src/")
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [8]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

In [ ]:
rag_system_prompt = """Answer the following questions as best you can. You may ask the human to use the following tools:

{tools}

IMPORTANT MEMORY TOOL REQUIREMENTS:
1. You MUST call get_memory tool FIRST, before calling any other tools
2. You MUST use user_id "user_12" for all memory operations
3. You MUST include ALL required parameters when calling memory tools
4. When calling add_memory or get_memory, you MUST use the exact format as below, don't include any other content,
    and make sure the input is a valid JSON object.

For get_memory tool, you MUST use this exact format:
{{
    "query": "user preferences",
    "top_k": 1,
    "user_id": "user_12"
}}

For add_memory tool, you MUST use this exact format:
{{
    "conversation": [
        {{
            "role": "user",
            "content": "Hi, I'm Alex. I'm looking for a trip to New York"
        }},
        {{
            "role": "assistant",
            "content": "Hello Alex! I've noted you are looking for a trip to New York."
        }}
    ],
    "user_id": "user_12",
    "metadata": {{
        "key_value_pairs": {{
            "type": "travel",
            "relevance": "high"
        }}
    }},
    "memory": "User is looking for a trip to New York."
}}

You may respond in one of two formats.
Use the following format exactly to ask the human to use a tool:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action (if there is no required input, include "Action Input: None")
Observation: wait for the human to respond with the result from the tool, do not assume the response

... (this Thought/Action/Action Input/Observation can repeat N times. If you do not need to use a tool, or after asking the human to use any tools and waiting for the human to respond, you might know the final answer.)
Use the following format once you have the final answer:

Thought: I now know the final answer
Final Answer: the final answer to the original input question
"""  # noqa: E501


In [ ]:
from pydantic import HttpUrl

from nat.embedder.nim_embedder import NIMEmbedderModelConfig
from nat.llm.nim_llm import NIMModelConfig
from nat.plugins.mem0ai.memory import Mem0MemoryClientConfig
from nat.plugins.mem0ai.memory import mem0_memory_client
from nat.retriever.milvus.register import MilvusRetrieverConfig
from nat.retriever.milvus.register import milvus_retriever_client
from nat.tool.memory_tools.add_memory_tool import AddToolConfig
from nat.tool.memory_tools.add_memory_tool import add_memory_tool
from nat.tool.memory_tools.get_memory_tool import GetToolConfig
from nat.tool.memory_tools.get_memory_tool import get_memory_tool
from nat.tool.retriever import RetrieverConfig
from nat.tool.retriever import retriever_tool
from nat.utils.sdk.nat_agent import NatReactAgent
from nat.utils.sdk.nat_embedder import NatEmbedder
from nat.utils.sdk.nat_function import NatFunction
from nat.utils.sdk.nat_llm import NatLLM
from nat.utils.sdk.nat_memory import NatMemory
from nat.utils.sdk.nat_retriever import NatRetriever

llm = NatLLM(
    config=NIMModelConfig(model_name="nvdev/meta/llama-3.3-70b-instruct", temperature=0, max_tokens=4096, top_p=1),
    name="nim_llm",
)
milvus_embedder = NatEmbedder(
    config=NIMEmbedderModelConfig(model_name="nvidia/nv-embedqa-e5-v5", truncate="END"),
    name="milvus_embedder",
)
memory = NatMemory(config=Mem0MemoryClientConfig(), name="saas_memory", memory=mem0_memory_client)

cuda_retriever = NatRetriever(config=MilvusRetrieverConfig(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="cuda_docs",
    embedding_model=milvus_embedder.embedder_name,
    top_k=10,
),
                              name="cuda_retriever",
                              retriever=milvus_retriever_client)

mcp_retriever = NatRetriever(config=MilvusRetrieverConfig(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="mcp_docs",
    embedding_model=milvus_embedder.embedder_name,
    top_k=10,
),
                             name="mcp_retriever",
                             retriever=milvus_retriever_client)

cuda_retriever_tool = NatFunction(
    config=RetrieverConfig(retriever=cuda_retriever.retriever_name,
                           topic="Retrieve documentation for NVIDIA's CUDA library"),
    function=retriever_tool,
    name="cuda_retriever_tool",
)
mcp_retriever_tool = NatFunction(
    config=RetrieverConfig(retriever=mcp_retriever.retriever_name,
                           topic="Retrieve information about Model Context Protocol (MCP)"),
    function=retriever_tool,
    name="mcp_retriever_tool",
)
add_memory_nat_tool = NatFunction(
    config=AddToolConfig(
        description=
        """Add any facts about user preferences to long term memory. Always use this if users mention a preference.
The input to this tool should be a string that describes the user's preference, not the question or answer.""",
        memory=memory.memory_name),
    function=add_memory_tool,
    name="add_memory_tool",
)
get_memory_nat_tool = NatFunction(
    config=GetToolConfig(
        description="""Always call this tool before calling any other tools, even if the user does not mention to use it.
The question should be about user preferences which will help you format your response.
For example: "How does the user like responses formatted?""",  # noqa: E501
        memory=memory.memory_name),
    function=get_memory_tool,
    name="get_memory_tool",
)

agent = NatReactAgent(
    tools=[cuda_retriever_tool, mcp_retriever_tool, add_memory_nat_tool, get_memory_nat_tool],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
    system_prompt=rag_system_prompt,
    memory=memory,
    retrievers=[cuda_retriever, mcp_retriever],
    referenced_embedders=[milvus_embedder],
)

In [11]:
await agent.prompt('How do I install CUDA?')

'To install CUDA, follow the steps outlined above, and make sure to consult the official NVIDIA CUDA documentation for detailed instructions tailored to your specific system configuration.'

In [12]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
agent.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  cuda_retriever_tool:
    retriever: cuda_retriever
    topic: Retrieve documentation for NVIDIA's CUDA library
    _type: nat_retriever
  mcp_retriever_tool:
    retriever: mcp_retriever
    topic: Retrieve information about Model Context Protocol (MCP)
    _type: nat_retriever
  add_memory_tool:
    description: |-
      Add any facts about user preferences to long term memory. Always use this if users mention a preference.
      The input to this tool should be a string that describes the user's preference, not the question or answer.
    memory: saas_memory
    _type: add_memory
  get_memory_tool:
    description: |-
      Always call this tool before calling any other tools, even if the user does not mention to use it.
      The question should be about user preferences which will help you format your response.
      For example: "How does the user like responses formatted?
    memory: saas_memory
    _type: get_memory
llms:
  nim_llm:
    model: nvdev/meta/llama-3.3-7